In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Organization")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())

In [ ]:
silver_base_path = "../../data_lake/silver/silver_organization/"
gold_base_path = "../../data_lake/gold/dim_organization/"
gold_dimdate = "../../data_lake/gold/dim_date/"
gold_dimtime = "../../data_lake/gold/dim_time/"
gold_dimpatient = "../../data_lake/gold/dim_patient/"

In [ ]:
df_patient = spark.read.format("parquet").load(gold_dimpatient)
df_dimdate = spark.read.format("parquet").load(gold_dimdate)
df_dimtime = spark.read.format("parquet").load(gold_dimtime)


In [ ]:
df_organization = (spark.read.format("parquet").load(silver_base_path))

In [ ]:
df_organization.show(5)

In [ ]:
df_organization.printSchema()

In [ ]:
df_inter = (df_organization.select(
        F.conv(F.substring(F.md5(col("organization_id")),1, 15), 16, 10).cast("bigint").alias("organization_key"),
        col("organization_id"),
        col("name"),
        col("type_code"),
        col("type_display"),
        col("is_active"),
        col("telecom_phone"),
        col("telecom_email"),
        col("telecom_fax"),
        col("address_line"),
        col("city"),
        col("state"),
        col("postalCode"),
        col("country"),
        col("utilization_encounters"),
        col("utilization_procedures"),
        col("utilization_labs"),
        col("utilization_prescriptions"),
        F.current_timestamp().alias("gold_timestamp")

    )
)

In [ ]:
df_inter.show(5)

In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)

In [ ]:
spark.stop()